In [2]:
!pip install dagshub mlflow -q
import dagshub
import mlflow
import pandas as pd
import numpy as np
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("DAGSHUB_TOKEN")

dagshub.init(repo_owner='kvizhmena11', repo_name='ML-assignment2-IEEE-Fraud-Detection', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow")
mlflow.set_experiment("LightGBM_Training")
print("✅ Connected to DagsHub!")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6872deca-30b9-4fa9-8842-35b409d140d9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4af8d0ff8f5fd1591bfed6d8f7023b998382e4b9ab528ddac81c4c50c177fd75




Output()

Accessing as kvizhmena11

Initialized MLflow to track repo "kvizhmena11/ML-assignment2-IEEE-Fraud-Detection"

Repository kvizhmena11/ML-assignment2-IEEE-Fraud-Detection initialized!

2026/05/07 10:09:52 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_Training' does not exist. Creating a new experiment.


✅ Connected to DagsHub!


# 🧹 Cleaning

In [3]:
import os
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

mlflow.end_run()
base_path = '/kaggle/input/competitions/ieee-fraud-detection/'

with mlflow.start_run(run_name="LightGBM_Cleaning"):
    train_transaction = pd.read_csv(os.path.join(base_path, 'train_transaction.csv'))
    train_identity = pd.read_csv(os.path.join(base_path, 'train_identity.csv'))
    
    train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    del train_transaction, train_identity
    
    nan_count = train.isna().sum().sum()
    fraud_rate = train['isFraud'].mean()
    
    mlflow.log_metric("initial_nan_count", nan_count)
    mlflow.log_metric("fraud_rate_pct", fraud_rate * 100)
    mlflow.log_metric("class_imbalance_ratio", (1 - fraud_rate) / fraud_rate)
    
    train = reduce_mem_usage(train)
    
    mlflow.log_param("total_rows", len(train))
    mlflow.log_param("total_columns", len(train.columns))
    print(f"✅ Dataset shape: {train.shape}")
    print(f"Fraud rate: {fraud_rate:.4f}")

Mem. usage decreased to 1044.70 Mb (46.6% reduction)
✅ Dataset shape: (590540, 434)
Fraud rate: 0.0350
🏃 View run LightGBM_Cleaning at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/6ff6412b7a5b4ab8906c25dbcdbde00b
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


# ⚙️ Feature Engineering

In [4]:
with mlflow.start_run(run_name="LightGBM_Feature_Engineering"):
    
    # Email domain grouping
    emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum',
              'outlook.com': 'microsoft', 'msn.com': 'microsoft', 'hotmail.com': 'microsoft',
              'live.com': 'microsoft', 'icloud.com': 'apple', 'mac.com': 'apple'}
    
    for c in ['P_emaildomain', 'R_emaildomain']:
        train[c + '_bin'] = train[c].map(emails)
        train[c + '_suffix'] = train[c].map(lambda x: str(x).split('.')[-1])
    
    # Time features
    train['Trans_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Trans_day_of_week'] = (train['TransactionDT'] // (3600 * 24)) % 7
    
    # Frequency encoding
    for col in ['card1', 'card2', 'card3', 'card5']:
        if col in train.columns:
            freq = train[col].astype(np.float32).value_counts(normalize=True)
            train[f'{col}_freq'] = train[col].astype(np.float32).map(freq)
    
    # Transaction amount features
    train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
    train['TransactionAmt_decimal'] = ((train['TransactionAmt'].astype(np.float32) - train['TransactionAmt'].astype(int)) * 1000).astype(int)
    
    # Group aggregations
    train['card1_TransactionAmt_mean'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('mean')
    train['card1_TransactionAmt_std'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('std')
    train['amt_vs_card_mean'] = train['TransactionAmt'] / (train['card1_TransactionAmt_mean'] + 1)
    train['TransactionAmt_zscore'] = (train['TransactionAmt'] - train['card1_TransactionAmt_mean']) / (train['card1_TransactionAmt_std'] + 1)
    
    if 'addr1' in train.columns and 'addr2' in train.columns:
        train['addr_match'] = (train['addr1'] == train['addr2']).astype(int)
    
    # Label encoding
    from sklearn.preprocessing import LabelEncoder
    cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3',
                'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23',
                'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35',
                'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo',
                'P_emaildomain_bin', 'P_emaildomain_suffix', 'R_emaildomain_bin', 'R_emaildomain_suffix']
    
    for col in cat_cols:
        if col in train.columns:
            le = LabelEncoder()
            train[col] = le.fit_transform(train[col].astype(str))
    
    train = train.copy()
    
    mlflow.log_param("total_engineered_features", 8)
    mlflow.log_param("feature_types", "freq_encoding, log_transform, group_aggregations, interactions")
    print(f"✅ Shape after FE: {train.shape}")

/tmp/ipykernel_57/2829436395.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[c + '_bin'] = train[c].map(emails)
/tmp/ipykernel_57/2829436395.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[c + '_suffix'] = train[c].map(lambda x: str(x).split('.')[-1])
/tmp/ipykernel_57/2829436395.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 

✅ Shape after FE: (590540, 451)
🏃 View run LightGBM_Feature_Engineering at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/9fca6984154d44dfaa9f8e1c4c447efc
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


# 🎯 Feature Selection


In [5]:
with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    
    # Drop high null and constant columns
    threshold = 0.90
    null_cols = [c for c in train.columns if train[c].isnull().sum() / len(train) > threshold]
    one_value_cols = [c for c in train.columns if train[c].nunique() <= 1]
    cols_to_drop = list(set(null_cols + one_value_cols))
    for c in ['isFraud', 'TransactionID', 'TransactionDT']:
        if c in cols_to_drop:
            cols_to_drop.remove(c)
    train.drop(cols_to_drop, axis=1, inplace=True)
    mlflow.log_param("cols_dropped_null_or_constant", len(cols_to_drop))
    
    # Importance-based selection on 10% sample
    X_fs = train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
    y_fs = train['isFraud']
    
    sample_idx = X_fs.sample(frac=0.1, random_state=42).index
    X_sample = X_fs.loc[sample_idx]
    y_sample = y_fs.loc[sample_idx]
    
    quick_model = LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=-1)
    quick_model.fit(X_sample, y_sample)
    
    importances = pd.Series(quick_model.feature_importances_, index=X_fs.columns)
    selected_features = importances[importances > 0].index.tolist()
    removed_low_importance = len(X_fs.columns) - len(selected_features)
    
    mlflow.log_param("selection_method", "lgbm_importance_on_10pct_sample")
    mlflow.log_param("importance_threshold", 0)
    mlflow.log_param("features_removed_low_importance", removed_low_importance)
    mlflow.log_param("final_selected_features", len(selected_features))
    
    print(f"✅ Selected {len(selected_features)} features out of {len(X_fs.columns)}")

✅ Selected 312 features out of 437
🏃 View run LightGBM_Feature_Selection at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/03fa66b77550494e8d2528e45536434a
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


# 🚀 Training


In [6]:
X = train[selected_features]
y = train['isFraud']
split_idx = int(len(X) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

Train: 472432 | Val: 118108


In [7]:
with mlflow.start_run(run_name="LightGBM_Underfit"):
    params = {'n_estimators': 50, 'max_depth': 2, 'learning_rate': 0.3,
              'random_state': 42, 'verbose': -1}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "underfitting_demonstration")
    
    m = LGBMClassifier(**params)
    m.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[UNDERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")

[UNDERFIT] Train: 0.8856 | Val: 0.8676 | Gap: 0.0180
🏃 View run LightGBM_Underfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/1194bba976214efc917406b3b12be4c1
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


In [9]:
with mlflow.start_run(run_name="LightGBM_Overfit"):
    params = {'n_estimators': 1000, 'max_depth': 15, 'learning_rate': 0.3,
              'subsample': 1.0, 'colsample_bytree': 1.0,
              'random_state': 42, 'verbose': -1}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "overfitting_demonstration")
    
    m = LGBMClassifier(**params)
    m.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[OVERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")


[OVERFIT] Train: 0.8891 | Val: 0.8118 | Gap: 0.0772
🏃 View run LightGBM_Overfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/1367910b26df48aab37dd36af995ee1c
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


In [10]:
# Best
with mlflow.start_run(run_name="LightGBM_Best"):
    params = {
        'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 1.0, 'reg_lambda': 5.0,
        'min_child_samples': 20,
        'scale_pos_weight': 30,
        'random_state': 42, 'verbose': -1
    }
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "best_balanced_model")
    
    m = LGBMClassifier(**params)
    m.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[BEST] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")

[BEST] Train: 0.9642 | Val: 0.9112 | Gap: 0.0530
🏃 View run LightGBM_Best at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/98ae5504491742149d722743c4889537
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1


# 💾 Pipeline & Model Registry


In [11]:
import mlflow.sklearn
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

best_params = {
    'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 5.0,
    'min_child_samples': 20, 'scale_pos_weight': 30,
    'random_state': 42, 'verbose': -1
}

best_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-999)),
    ('classifier', LGBMClassifier(**best_params))
])

best_pipeline.fit(X_train, y_train)

with mlflow.start_run(run_name="LightGBM_Final_Pipeline"):
    train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, best_pipeline.predict_proba(X_val)[:,1])
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    mlflow.log_param("selected_features_count", len(selected_features))
    
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="lgbm_pipeline",
        registered_model_name="LightGBM_Fraud_Pipeline",
        input_example=X_val.iloc[:5]
    )
    
    print(f"✅ Pipeline saved to Model Registry!")
    print(f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

with open('selected_features_lgbm.json', 'w') as f:
    json.dump(selected_features, f)
mlflow.log_artifact('selected_features_lgbm.json')

mlflow.end_run()
print("✅ LightGBM notebook complete!")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/07 10:22:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 10:22:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains i

✅ Pipeline saved to Model Registry!
Train AUC: 0.9638 | Val AUC: 0.9101
🏃 View run LightGBM_Final_Pipeline at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/d61b1c5f36d944cf8a292c2e8f3ffcea
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1
🏃 View run fun-finch-685 at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1/runs/54aa0fadebe340f9ad7183bd723e60f2
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/1
✅ LightGBM notebook complete!
